
### Issue Encountered: Data Type Mismatch in Bulk Parquet Read

When reading multiple Parquet files in a single operation, we encountered errors due to columns having different datatypes across files (e.g., `increment` as `int` in one file and `float` in another). This caused a data type mismatch during the merge into the target table.

#### Resolution

To resolve this, we processed each file individually:
- Read each file separately.
- Compared column datatypes against the target table schema.
- Cast mismatched columns to the target datatype before merging.

This approach ensured successful data integration without errors.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE  dev_ws_1.poc.target_1 (
  id INT,
  name STRING,
  age INT,
  increment FLOAT
)
""")

In [0]:
spark.sql("""
INSERT INTO dev_ws_1.poc.target_1 (id, name, age, increment) VALUES
  (1, 'Alice', 30, 1.5),
  (2, 'Bob', 25, 2.0),
  (3, 'Charlie', 35, 0.5)
""")

In [0]:
from pyspark.sql import Row

dbutils.fs.rm('/Volumes/dev_ws_1/poc/testdata/raw/file1', True)
dbutils.fs.rm('/Volumes/dev_ws_1/poc/testdata/raw/file2', True)

# Data with increment as integer
data_int = [
    Row(id=4, name='David', age=28, increment=3),
    Row(id=5, name='Eva', age=32, increment=4)
]
df_int = spark.createDataFrame(data_int)
df_int.write.mode('overwrite').parquet('/Volumes/dev_ws_1/poc/testdata/raw/file1')

# Data with increment as float
data_float = [
    Row(id=6, name='Frank', age=29, increment=2.7),
    Row(id=7, name='Grace', age=31, increment=3.3)
]
df_float = spark.createDataFrame(data_float)
df_float.write.mode('overwrite').parquet('/Volumes/dev_ws_1/poc/testdata/raw/file2')

In [0]:
import os

raw_path = "/Volumes/dev_ws_1/poc/testdata/raw/"
files = dbutils.fs.ls(raw_path)

target_schema = spark.table("dev_ws_1.poc.target_1").schema

try:
    df_all = spark.read.option("recursiveFileLookup", "true").parquet(raw_path)
    display(df_all)
    df_all.createOrReplaceTempView("staging")
    print("Created temp view 'staging' for all files")
    spark.sql("""
    MERGE INTO dev_ws_1.poc.target_1 t
    USING staging s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print("MERGE completed for all files without errors")
except Exception as e:
    err_msg = str(e).split('\n')[0].strip()
    print(f"Exception occurred during bulk merge: {err_msg}")
    if "Data type mismatches" in str(e):
        print("Processing files individually due to data type mismatch...")
        for file in files:
            print(f"Processing file: {file.path}")
            try:
                df = spark.read.parquet(file.path)
                display(df)
                df.createOrReplaceTempView("staging")
                print("Created temp view 'staging'")
                spark.sql("""
                MERGE INTO dev_ws_1.poc.target_1 t
                USING staging s
                ON t.id = s.id
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
                """)
                print("MERGE completed without errors")
            except Exception as e_file:
                err_msg_file = str(e_file)
                print(f"Exception occurred: {err_msg_file}")
                if "Data type mismatches" in err_msg_file:
                    mismatched_cols = []
                    for field in target_schema:
                        if f"Column '{field.name}'" in err_msg_file or f"column {field.name}" in err_msg_file:
                            mismatched_cols.append(field.name)
                    print(f"Mismatched columns identified: {mismatched_cols}")
                    df = spark.read.option("mergeSchema", "true").parquet(file.path)
                    for col in mismatched_cols:
                        target_type = [f.dataType for f in target_schema if f.name == col][0]
                        print(f"Casting column '{col}' to target type '{target_type}'")
                        df = df.withColumn(col, df[col].cast(str(target_type)))
                    display(df)
                    df.createOrReplaceTempView("staging")
                    print("Created temp view 'staging' after casting")
                    spark.sql("""
                    MERGE INTO dev_ws_1.poc.target_1 t
                    USING staging s
                    ON t.id = s.id
                    WHEN MATCHED THEN UPDATE SET *
                    WHEN NOT MATCHED THEN INSERT *
                    """)
                    print("MERGE completed after handling mismatched columns")
                else:
                    print("Unhandled exception, raising...")
                    raise
    else:
        print("Unhandled exception, raising...")
        raise

In [0]:
%sql
select * from dev_ws_1.poc.target_1